## Task 1

In [4]:
# %pip install -r requirements.txt

## Task 2

In [5]:
# !pip install openmeteo-requests
# !pip install requests-cache retry-requests numpy pandas

In [6]:
# %pip install "urllib3<2" requests-cache retry-requests openmeteo-requests 

In [7]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 39.7756,
	"longitude": 47.6186,
	"start_date": "2025-04-19",
	"end_date": "2026-04-19",
	"daily": ["temperature_2m_mean", "temperature_2m_max", "sunshine_duration", "precipitation_sum", "wind_speed_10m_max", "et0_fao_evapotranspiration_sum", "temperature_2m_min", "soil_temperature_0_to_7cm_mean"],
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()
daily_et0_fao_evapotranspiration_sum = daily.Variables(5).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(6).ValuesAsNumpy()
daily_soil_temperature_0_to_7cm_mean = daily.Variables(7).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["soil_temperature_0_to_7cm_mean"] = daily_soil_temperature_0_to_7cm_mean

historical_beylagan = pd.DataFrame(data = daily_data)
print("\nBeylagan\n", historical_beylagan) 

Coordinates: 39.82425308227539°N 47.6323127746582°E
Elevation: 61.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                           date  temperature_2m
0    2025-04-19 00:00:00+00:00            6.25
1    2025-04-19 01:00:00+00:00            5.95
2    2025-04-19 02:00:00+00:00            5.50
3    2025-04-19 03:00:00+00:00            6.35
4    2025-04-19 04:00:00+00:00            9.05
...                        ...             ...
8779 2026-04-19 19:00:00+00:00           15.45
8780 2026-04-19 20:00:00+00:00           15.70
8781 2026-04-19 21:00:00+00:00           15.35
8782 2026-04-19 22:00:00+00:00           15.05
8783 2026-04-19 23:00:00+00:00           14.35

[8784 rows x 2 columns]

Beylagan
                          date  temperature_2m_mean  temperature_2m_max  \
0   2025-04-19 00:00:00+00:00            12.779168           19.400000   
1   2025-04-20 00:00:00+00:00            12.256249           17.950001   
2   2025-04-21 00:00:00+00:00            15.533336       

In [8]:
data_beylagan

NameError: name 'data_beylagan' is not defined

In [ ]:
historical_beylagan['region'] = 'Beylagan'

In [ ]:
historical_beylagan

In [ ]:
# the main headings in the structure (dictionary)
hist_cols = set(historical_beylagan.columns)
hist_cols

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(historical_beylagan['date'], historical_beylagan['temperature_2m_max'], color='orangered', label='Max Temp')

plt.title('Daily Maximum Temperature (1 Year)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.show()

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 39.7756,
	"longitude": 47.6186,
	"daily": ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "sunshine_duration", "precipitation_sum", "wind_speed_10m_max", "et0_fao_evapotranspiration_sum"],
	"hourly": "temperature_2m",
	"past_days": 0,
	"forecast_days": 7,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(2).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(3).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(4).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(5).ValuesAsNumpy()
daily_et0_fao_evapotranspiration_sum = daily.Variables(6).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum

forecast_beylagan = pd.DataFrame(data = daily_data)
print("\nDaily data\n", forecast_beylagan)

In [ ]:
forecast_beylagan['region'] = 'Beylagan'

In [ ]:
forecast_beylagan

In [ ]:
hist_cols = set(historical_beylagan.columns)
fore_cols = set(forecast_beylagan.columns)

print("Those which are in Historical and not in Forecast:", hist_cols - fore_cols)
print("\nThose in Forecast and not in Historical:", fore_cols - hist_cols)
print("\nCommon areas in both:", hist_cols.intersection(fore_cols))

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 39.7756,
	"longitude": 47.6186,
	"start_date": "2025-04-19",
	"end_date": "2026-04-19",
	"daily": ["temperature_2m_mean", "temperature_2m_max", "sunshine_duration", "precipitation_sum", "wind_speed_10m_max", "et0_fao_evapotranspiration_sum", "temperature_2m_min", "soil_temperature_0_to_7cm_mean", "soil_moisture_0_to_7cm_mean", "relative_humidity_2m_mean", "shortwave_radiation_sum"],
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()
daily_et0_fao_evapotranspiration_sum = daily.Variables(5).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(6).ValuesAsNumpy()
daily_soil_temperature_0_to_7cm_mean = daily.Variables(7).ValuesAsNumpy()
daily_soil_moisture_0_to_7cm_mean = daily.Variables(8).ValuesAsNumpy()
daily_relative_humidity_2m_mean = daily.Variables(9).ValuesAsNumpy()
daily_shortwave_radiation_sum = daily.Variables(10).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["et0_fao_evapotranspiration_sum"] = daily_et0_fao_evapotranspiration_sum
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["soil_temperature_0_to_7cm_mean"] = daily_soil_temperature_0_to_7cm_mean
daily_data["soil_moisture_0_to_7cm_mean"] = daily_soil_moisture_0_to_7cm_mean
daily_data["relative_humidity_2m_mean"] = daily_relative_humidity_2m_mean
daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum

data_beylagan_updated = pd.DataFrame(data = daily_data)
print("\nDaily data\n", data_beylagan_updated) 

In [ ]:
data_beylagan_updated['region'] = 'Beylagan'

In [ ]:
data_beylagan_updated

In [ ]:
print(f"--- Weather Variable Documentation (Location: {params['latitude']}, {params['longitude']}) ---")
print("1. Temperature 2m Max: The maximum temperature recorded during the day at a height of 2m. Unit: °C")
print("2. Mean Relative Humidity (2m): Daily average of air humidity. Unit: %")
print("3. Mean Soil Moisture (0-7cm):Volume fraction of water in the topsoil. Unit: m³/m³")
print("4. Reference Evapotranspiration (ET0): The amount of water evaporated from soil and plants. Unit: mm")
print("-" * 55) 